# HW1 語言模型初體驗

> 課程：生成式人工智慧：理論與實務（逢甲大學 115-1）　說明：<https://github.com/chang-ye-tu/genai/blob/main/hw/hw1.md>
> 核心段落：第 0–4 節（實作測驗只出這些段落的題目）；第 5、6 節為選做，不計分。
> 做法：**執行階段 → 變更執行階段類型 → T4 GPU**，由上而下逐格執行；看到「✍️ 請回答」就把觀察寫進該文字格。全部跑完後「檔案 → 下載 → .ipynb」上傳 iLearn，再作答實作測驗。
> **請勿更改模型名稱、版本、隨機種子與資料檔**，否則實作測驗的數值題會對不上。


In [ ]:
# @title 第 0 節：安裝與載入模型（第一次約 1–2 分鐘）
%pip -q install transformers==5.16.1 accelerate==1.14.0
import torch, transformers
USE_SMALL = False  # @param {type:"boolean"}  Colab 額度不足時改 True（實作測驗的數值題請以 1.5B 為準）
MODEL, REV = ("Qwen/Qwen2.5-0.5B-Instruct", "7ae557604adf67be50417f59c2c2f167def9a775") if USE_SMALL else ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306")
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL, revision=REV)
DTYPE = torch.float32 if "fp32" == "fp32" else (torch.float16 if device == "cuda" else torch.float32)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=DTYPE).to(device).eval()
print("模型：", MODEL, "| 裝置：", device, "| dtype：", DTYPE)
import platform, importlib.metadata as _meta
def _v(p):
    try: return _meta.version(p)
    except Exception: return "missing"  # metadata 查不到時印 missing；若同一格更早的 import 已失敗，程式到不了這裡，check_submissions 會判「無版本資訊／執行錯誤」
print("VERSIONS", "python=" + platform.python_version(), "torch=" + torch.__version__, *[p + "=" + _v(p) for p in ["transformers", "accelerate"]])
print("詞彙表大小（tokenizer）：", len(tok), "| eos token：", tok.eos_token, tok.eos_token_id, "| pad：", tok.pad_token)

def chat_prompt(user, system=None, history=None):
    """把訊息套上對話模板，回傳模型實際看到的字串。"""
    msgs = []
    if system is not None:
        msgs.append({"role": "system", "content": system})
    for u, a in (history or []):
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": user})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(prompt, max_new_tokens=64, temperature=0.0, top_p=1.0, raw=False):
    """raw=True 表示 prompt 已是完整字串（不套模板）。temperature=0 代表 greedy。"""
    text = prompt if raw else chat_prompt(prompt)
    ids = tok(text, return_tensors="pt").to(device)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        kw.update(do_sample=False)
    out = model.generate(**ids, **kw)
    return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()


## 第 1 節 分詞（tokenizer）：文字怎麼變成 token

模型看到的不是字，而是 token。下面對 15 個字串分詞，觀察：中文、英文、數字、符號各怎麼切？哪一種最「耗 token」？


In [ ]:
strings = {
 "S1": "逢甲大學", "S2": "生成式人工智慧：理論與實務", "S3": "Feng Chia University",
 "S4": "臺灣最高的山是哪座？", "S5": "Large language models predict the next token.",
 "S6": "人工智慧", "S7": "Artificial intelligence", "S8": "1234567890",
 "S9": "我今天早上在逢甲夜市吃了一碗大腸麵線，然後去圖書館讀生成式人工智慧的講義。",
 "S10": "This morning I ate a bowl of noodles at Feng Chia Night Market, then went to the library to read the generative AI lecture notes.",
 "S11": "ChatGPT", "S12": "generative", "S13": "生成式", "S14": "！！！！！！！！！！", "S15": "2026年9月14日",
}
for k, s in strings.items():
    ids = tok.encode(s, add_special_tokens=False)
    print(f"{k:>3} {len(ids):>3} tokens | {s!r}")
    print("      ", [tok.decode([i]) for i in ids])


In [ ]:
# 自己試試看：換成你的名字、你的系所、一段英文歌詞……
my_text = "在這裡輸入你想測的文字"  # @param {type:"string"}
ids = tok.encode(my_text, add_special_tokens=False)
print(len(ids), "tokens:", [tok.decode([i]) for i in ids])


✍️ **請回答 1-1**：中文、英文、數字三種文字中，哪一種每個字元平均消耗的 token 最多？這對「API 依 token 計費」與「上下文長度上限」各有什麼影響？

（在這裡作答）


## 第 2 節 下一個 token 的機率分佈（fp32）

語言模型真正輸出的是「下一個 token 的機率分佈」。比較兩種輸入：
- **raw**：直接把「臺灣最高的山是」當成未完成的句子；
- **chat**：把「臺灣最高的山是哪座？」套上對話模板（模型實際看到 system prompt + 使用者訊息）。


In [ ]:
@torch.no_grad()
def top5(text, raw=False, k=5):
    s = text if raw else chat_prompt(text)
    ids = tok(s, return_tensors="pt").to(device)
    logits = model(**ids).logits[0, -1].float()
    p = torch.softmax(logits, -1)
    v, i = p.topk(k)
    return [(tok.decode([int(j)]), round(float(x), 3)) for x, j in zip(v, i)]

print("chat 「臺灣最高的山是哪座？」 →", top5("臺灣最高的山是哪座？"))
print("raw  「臺灣最高的山是」     →", top5("臺灣最高的山是", raw=True))
print("raw  「水的沸點是攝氏」     →", top5("水的沸點是攝氏", raw=True))
print("raw  「在 0.5 大氣壓下，水的沸點是攝氏」 →", top5("在 0.5 大氣壓下，水的沸點是攝氏", raw=True))
print("raw  「黃色的」             →", top5("黃色的", raw=True))


In [ ]:
# greedy（temperature=0）把整句生完：這個 1.5B 小模型答對了嗎？
print("chat greedy →", generate("臺灣最高的山是哪座？", max_new_tokens=40))
print("raw  greedy →", generate("臺灣最高的山是", max_new_tokens=20, raw=True))


✍️ **請回答 2-1**：raw 與 chat 兩種輸入的前 5 名為什麼差這麼多？模型的答案正確嗎（臺灣最高峰是玉山，3,952 公尺）？用第 1 單元「文字接龍」與「世界知識」的說法解釋。

（在這裡作答）


## 第 3 節 抽樣與 temperature

同一個提示，比較 greedy、temperature 0.7、1.5，以及 top-p。每種各生成 3 次。


In [ ]:
prompt = "用一句話介紹逢甲大學。"
for temp in [0.0, 0.7, 1.5]:
    print(f"=== temperature = {temp} ===")
    for r in range(3):
        print(f"[{r+1}]", generate(prompt, max_new_tokens=48, temperature=temp))
print("=== temperature 0.7, top_p 0.5 ===")
for r in range(3):
    print(f"[{r+1}]", generate(prompt, max_new_tokens=48, temperature=0.7, top_p=0.5))


✍️ **請回答 3-1**：greedy 的三次一樣嗎？溫度 1.5 的輸出出現了什麼問題？如果你要做「客服回覆」與「寫詩」，各會選什麼設定？

（在這裡作答）


## 第 4 節 對話模板與 system prompt：模型實際看到什麼

`apply_chat_template` 會把你的訊息包成模型訓練時的格式（ChatML），並自動加上**預設 system prompt**。


In [ ]:
p = chat_prompt("臺灣最高的山是哪座？")
print(p)
print("---- 整段輸入 token 數：", len(tok(p).input_ids), "（使用者問題本身：", len(tok.encode("臺灣最高的山是哪座？", add_special_tokens=False)), "）")


In [ ]:
# 改寫 system prompt：身分、語言、日期
sys1 = "你是逢甲大學的課程助教，只用繁體中文回答，回答不超過 30 字。"
print("A.", generate(chat_prompt("臺灣最高的山是哪座？", system=sys1), raw=True, max_new_tokens=40))
print("B. 沒有日期 →", generate("今天幾月幾號？", max_new_tokens=30))
print("C. 有日期  →", generate(chat_prompt("今天幾月幾號？", system="今天是 2026 年 9 月 21 日。"), raw=True, max_new_tokens=30))


In [ ]:
# 多輪對話：歷史要自己串進去
hist = [("臺灣最高的山是哪座？", "臺灣最高的山是玉山，海拔約 3,952 公尺。")]
print("有歷史 →", generate(chat_prompt("第二高的呢？", history=hist), raw=True, max_new_tokens=40))
print("無歷史 →", generate("第二高的呢？", max_new_tokens=40))


✍️ **請回答 4-1**：預設 system prompt 寫了什麼？「無歷史」時模型怎麼回答「第二高的呢？」？這和第 1 單元的「遺忘」有什麼關係？

（在這裡作答）


## 第 5 節（選做）提示策略小實驗（第 2 單元）

12 題小學算術／簡單邏輯題，比較三種提示：直接問、思維鏈（請一步一步思考）、少樣本（給 2 個範例）。每種跑 2 次（temperature 0.7）。


In [ ]:
import re
QA = [
 ("小明有 3 顆蘋果，又買了 5 顆，吃掉 2 顆，還剩幾顆？", 6),
 ("一本書 120 元，打八折後多少元？", 96),
 ("教室有 4 排座位，每排 7 個，坐了 25 人，還有幾個空位？", 3),
 ("從 1 加到 10 的總和是多少？", 55),
 ("一個長方形長 8 公分、寬 5 公分，面積是多少平方公分？", 40),
 ("小華每天存 15 元，存 12 天共多少元？", 180),
 ("現在是上午 9 點，再過 5 小時是下午幾點？", 2),
 ("48 顆糖平均分給 6 個人，每人幾顆？", 8),
 ("一打雞蛋 12 顆，買 3 打再用掉 10 顆，剩幾顆？", 26),
 ("某數的 3 倍是 27，某數是多少？", 9),
 ("公車上有 18 人，到站下車 7 人、上車 4 人，車上有幾人？", 15),
 ("一週有 7 天，5 週又 3 天共幾天？", 38),
]
def direct(q): return q + "\n請只回答一個數字。"
def cot(q): return q + "\n請一步一步思考，最後一行寫「答案：數字」。"
def fewshot(q): return ("範例 1：小美有 2 枝筆，再買 3 枝，共幾枝？ 答案：5\n"
                        "範例 2：一箱 24 罐飲料，喝掉 9 罐，剩幾罐？ 答案：15\n"
                        f"問題：{q} 答案：")
def last_int(s):
    m = re.findall(r"-?\d+", s.replace(",", ""))
    return int(m[-1]) if m else None

NUM_RUNS = 2
results = {}
for name, f in [("直接問", direct), ("思維鏈", cot), ("少樣本", fewshot)]:
    accs = []
    for run in range(NUM_RUNS):
        correct = 0
        for q, a in QA:
            out = generate(f(q), max_new_tokens=160, temperature=0.7)
            correct += int(last_int(out) == a)
        accs.append(correct / len(QA))
    results[name] = accs
    print(f"{name}: " + "，".join(f"第 {k+1} 次 {acc:.0%}" for k, acc in enumerate(accs)))


✍️ **請回答 5-1**：哪種提示最好？兩次結果一樣嗎？如果要用這個實驗說服別人「思維鏈有效／無效」，還需要做什麼（想想第 6 單元的評量原則）？

（在這裡作答）


## 第 6 節（選做）模型規模：參數從哪裡來

從設定檔讀出結構，直接算參數量（不需要載入權重）。


In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL, revision=REV)
print({k: getattr(cfg, k) for k in ["hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "intermediate_size", "vocab_size", "max_position_embeddings", "tie_word_embeddings"]})
with torch.device("meta"):
    m = AutoModelForCausalLM.from_config(cfg)
total = sum(p.numel() for p in m.parameters())
emb = m.model.embed_tokens.weight.numel()
print(f"total params: {total:,}  ≈ {total/1e9:.2f} B")
print(f"token embedding: {emb:,}  ({emb/total:.1%})")


✍️ **請回答 6-1**：embedding 為什麼占這麼大比例？它的大小由哪兩個數字決定？如果詞彙表縮小一半，會發生什麼事？

（在這裡作答）


## ✍️ AI 使用聲明（必填）

| 項目 | 內容 |
|------|------|
| 使用的工具 | （例如：ChatGPT 免費版、Colab 內建 Gemini） |
| 用在哪些工作 | （例如：解釋錯誤訊息、幫我看懂某一格程式） |
| 我自己完成的部分 | （例如：全部執行、所有 ✍️ 回答） |
| 我如何驗證 AI 的說法 | （例如：實際執行、對照投影片） |

姓名／學號：
